# ScholarLens 02 - Retrieval + Question Answering (DistilBERT vs RoBERTa)
Owner: Member 2. Retriever comparison (MiniLM vs TF-IDF) -> QA baselines -> fine-tune both QA models -> pipeline test on real papers.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sajithaucm/Scholarlens_Project/blob/main/notebooks/02_qa_retrieval.ipynb)

## 0. Setup (run every time you open this notebook)
1. **Runtime -> Change runtime type -> T4 GPU** (Hardware accelerator).
2. The repo URL below already points to the team repo (sajithaucm/Scholarlens_Project).
3. Checkpoints are saved to **Google Drive** so they survive a disconnect; small result files are saved in the repo's `results/` folder and pushed to GitHub at the end.

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "https://github.com/sajithaucm/Scholarlens_Project.git"
if not os.path.exists("/content/Scholarlens_Project"):
    !git clone $REPO /content/Scholarlens_Project
%cd /content/Scholarlens_Project
!git pull -q
!pip install -q -r requirements.txt

# Checkpoints -> Google Drive (big files never go to GitHub)
os.environ["SCHOLARLENS_OUTPUT_DIR"] = "/content/drive/MyDrive/Scholarlens_Project/outputs"
os.makedirs(os.environ["SCHOLARLENS_OUTPUT_DIR"], exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze > results/environment.txt   # exact library versions, for reproducibility

## 1. Quick test

In [ ]:
!python -m src.qa.train --model distilbert-base-uncased --max-samples 200 --epochs 1

## 2. Retrieval evaluation (Recall@1, Recall@3, MRR on SQuAD 2.0 dev articles)

In [ ]:
!python -m src.qa.retriever --evaluate

## 3. QA baselines (always-no-answer, TF-IDF sentence)

In [ ]:
!python -m src.baselines --task qa

## 4. Fine-tune DistilBERT

In [ ]:
!python -m src.qa.train --model distilbert-base-uncased

## 5. Fine-tune RoBERTa-base

In [ ]:
!python -m src.qa.train --model roberta-base

## 6. Errors to analyse
Wrongly abstaining on answerable questions, answering unanswerable ones, partial spans.

In [ ]:
import json
rows = [json.loads(l) for l in open('results/qa_roberta-base_samples.jsonl')]
wrong = [r for r in rows if not r['correct']]
print(f'{len(wrong)}/{len(rows)} wrong in the sample')
for r in wrong[:8]:
    print('Q:', r['question'], '| gold:', r['gold'] or 'NO ANSWER', '| pred:', r['prediction'] or 'NO ANSWER')

## 7. Domain-shift test on real papers
Fill `eval/paper_qa.jsonl` and `eval/papers/*.txt` first (see `eval/README.md`). Run once per QA model:

In [ ]:
OUT = os.environ['SCHOLARLENS_OUTPUT_DIR']
!python -m src.evaluation.paper_qa --qa-model $OUT/qa/roberta-base/best
!python -m src.evaluation.paper_qa --qa-model $OUT/qa/distilbert-base-uncased/best

In [ ]:
!python -m src.compare

## Save results to GitHub
Results (`results/*.json`, figures, sample predictions) are small and must be committed.
Create a GitHub **fine-grained personal access token** (repo contents: read & write) and paste it when asked. It is not stored in the notebook.
Alternative: download the `results/` folder (Files panel) and commit it from VS Code.

In [ ]:
%cd /content/Scholarlens_Project
from getpass import getpass
name, email = input("Your name: "), input("Your GitHub email: ")
token = getpass("GitHub token: ")
!git config user.name "$name" && git config user.email "$email"
!git pull -q
!git add results/
!git commit -m "Add results from Colab run" || echo "nothing to commit"
!git push https://$token@github.com/sajithaucm/Scholarlens_Project.git HEAD:main
del token